In [1]:
!pip3 install pandas
!pip3 install numpy
!pip3 install scikit-learn
!pip3 install pyarrow
!pip3 install optuna
%pip install -U xgboost

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb


In [6]:
import os
pathKaggle = "../input/datasets/luisfilipelima/bancotransacao/"
path = "../../data/processed/"
pathDrive = "/content/drive/MyDrive/TreinosIAS/XGBoosting/dados/"
path=pathDrive
path = pathKaggle
path


'../input/datasets/luisfilipelima/bancotransacao/'

In [7]:
obt = pd.read_parquet(path+"obt_with_terminal_history.parquet")
obtTrain = pd.read_parquet(path+"obt_train.parquet")
obtTest = pd.read_parquet(path+"obt_validation.parquet")
obtTrain

,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud,...,card_distinct_terminals_24h,seconds_since_last_card_tx,terminal_tx_count_1h,terminal_tx_count_24h,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d
0,63551,2018-01-03 14:46:39,13.790000,30469057923212,98,FreshMart,14,3,393.0,False,...,0,NaN,1,22,22,3210.0,0,NaN,NaN,NaN
1,160731,2018-01-07 16:31:49,11.020000,30469057923212,955,Elegant Carpets,16,0,104.0,False,...,0,351910.0,2,31,28,160.0,1,13.790000,NaN,NaN
2,264046,2018-01-12 05:53:34,37.709999,30469057923212,854,Vitality Gym,5,5,137.0,False,...,0,393705.0,2,24,24,1170.0,2,12.405000,1.958685,12.919379
3,264993,2018-01-12 06:50:19,25.290001,30469057923212,973,Gold Star Car Wash,6,5,103.0,False,...,1,3405.0,1,30,30,1386.0,3,20.840000,14.675349,0.303230
4,373789,2018-01-16 13:06:01,18.660000,30469057923212,786,Starlight Cinema,13,2,165.0,False,...,0,368142.0,1,23,23,1953.0,4,21.952500,12.187202,-0.270160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3428899,3392898,2018-04-29 19:19:24,513.229980,4682776545183079364,1850,Elite Gaming,19,0,41.0,False,...,5,30208.0,0,22,22,4609.0,181,154.742320,164.837789,2.174791
3428900,3418568,2018-04-30 14:29:23,67.320000,4682776545183079364,931,FreshMart,14,1,226.0,False,...,1,68999.0,2,25,24,726.0,179,157.706536,167.560088,-0.539428
3428901,3419962,2018-04-30 15:08:18,56.209999,4682776545183079364,280,Creative Art Supplies,15,1,415.0,False,...,2,2335.0,2,22,22,1008.0,180,157.204389,167.227149,-0.603935
3428902,3420910,2018-04-30 15:37:27,244.830002,4682776545183079364,1850,Elite Gaming,15,1,42.0,False,...,3,1749.0,2,21,21,89.0,181,156.646408,166.930858,0.528264


In [15]:
cat_features = ["terminal_category"]

features = [
    "tx_amount",
    "hour",
    "day_week",
    "terminal_age_days",
    "terminal_fraud_rate",
    "card_fraud_rate",
    "card_distinct_terminals",
    "terminal_new_for_card",
    "card_tx_count_1h",
    "card_tx_count_24h",
    "card_distinct_terminals_24h",
    "seconds_since_last_card_tx",
    "terminal_tx_count_1h",
    "terminal_tx_count_24h",
    "terminal_distinct_cards_24h",
    "seconds_since_last_terminal_tx",
    "card_history_count_30d",
    "card_amount_mean_30d",
    "card_amount_std_30d",
    "amount_z_score_30d",
    "terminal_category",
]
target = "is_fraud"
def prepararParaXGBoostging(df):
    df["datetime"] = pd.to_datetime(df["tx_datetime"], errors="raise")
    df[target] = df[target].astype(int)

    # XGB possui um suporte nativo para dados categoricos, tem que definir o tipo do dado como "category"
    df["terminal_category"] = (
        df["terminal_category"]
        .astype("object")
        .fillna("SEM_CATEGORIA")
        .astype("category")
)
    df = df.sort_values("tx_datetime").reset_index(drop=True)
    return df


    
obt = prepararParaXGBoostging(obt)
obtTrain = prepararParaXGBoostging(obtTrain)
obtTest = prepararParaXGBoostging(obtTest)


In [17]:
def calcular_lucro(TransacoesAceitas):
    if TransacoesAceitas.empty:
        print("Nenhuma transação aceita; lucro = 0.")
        return 0.0
    legitimas = TransacoesAceitas["is_fraud"].eq(0)
    fraudulentas = TransacoesAceitas["is_fraud"].eq(1)

    receita = (
        0.03 * TransacoesAceitas.loc[legitimas, "tx_amount"].sum()
    )

    prejuizo = (
        TransacoesAceitas.loc[fraudulentas, "tx_amount"].sum()
    )
    print("CALCULANDO MELHOR LUCRO...")
    print(f"VALOR:{float(receita - prejuizo)}")
    # Recusas têm resultado financeiro direto igual a zero.
    return float(receita - prejuizo).__round__(2)


def classificarTransacoes(x,y,prob,limiar):
    transacoes = x.copy()
    transacoes["is_fraud"] = np.asarray(y, dtype=int)
    transacoes["probabilidade_fraude"] = prob

    aceita = transacoes["probabilidade_fraude"] < limiar

    TransacoesAceitas = transacoes.loc[aceita].copy()
    TransacoesRecusadas = transacoes.loc[~aceita].copy() # pega todas menos as acceitas

    return TransacoesAceitas, TransacoesRecusadas

def encontrar_melhor_limiar(modelo, X_val, y_val):
    probabilidades = modelo.predict_proba(X_val)[:, 1]
    valores = X_val["tx_amount"].to_numpy()
    y = np.asarray(y_val, dtype=int)

    # Ordena da menor para a maior chance de fraude. Assim cada posição
    # representa o lucro acumulado ao aceitar as transações até aquele ponto.
    ordem = np.argsort(probabilidades, kind="stable")
    probabilidades_ordenadas = probabilidades[ordem]
    valores_ordenados = valores[ordem]
    y_ordenado = y[ordem]

    lucro_por_transacao = np.where(
        y_ordenado == 0,
        0.03 * valores_ordenados,
        -valores_ordenados,
    )
    lucro_acumulado = np.cumsum(lucro_por_transacao)

    
    fim_de_grupo = np.concatenate([
        probabilidades_ordenadas[1:] != probabilidades_ordenadas[:-1],
        [True],
    ])
    indices = np.flatnonzero(fim_de_grupo)
    lucros = lucro_acumulado[indices]

    
    lucros = np.concatenate([[0.0], lucros])
    melhor_posicao = int(np.argmax(lucros))
    melhor_lucro = float(lucros[melhor_posicao])

    if melhor_posicao == 0:
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[0], -np.inf
        )
    else:
        indice = indices[melhor_posicao - 1]
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[indice], np.inf
        )

    return float(melhor_limiar), melhor_lucro


In [18]:
from sklearn.model_selection import train_test_split
def separTreino(obtTrain,evalSize = 0.2, limiarSize = 0.2):
    obtBusca,resto = train_test_split(obtTrain, test_size=evalSize, random_state=42)
    obtEvaluation,obtLimiar = train_test_split(resto, test_size=limiarSize, random_state=42)

    return obtBusca,obtEvaluation,obtLimiar

obtBusca,obtEvaluation,obtLimiar = separTreino(obtTrain)



obtTrain = obtTrain.loc[~obtTrain.index.isin(obtLimiar.index)].copy()


In [19]:
print(obtTrain.loc[obtTrain["is_fraud"] == 1].shape)
print(obtTrain.loc[obtTrain["is_fraud"] == 0].shape)
proporcao = obtTrain.loc[obtTrain["is_fraud"] == 1].shape[0] / obtTrain.shape[0]
print(proporcao)


(29750, 24)
(3261997, 24)
0.009037754116583079


In [20]:
Buscatrain = obtBusca.tail(300_000).copy()

print(obtTrain.loc[obtTrain["is_fraud"] == 1].shape)
print(obtTrain.loc[obtTrain["is_fraud"] == 0].shape)
proporcao = obtTrain.loc[obtTrain["is_fraud"] == 1].shape[0] / obtTrain.shape[0]
print(proporcao)

print(Buscatrain.loc[Buscatrain['is_fraud'] == 1].shape)
print(Buscatrain.loc[Buscatrain['is_fraud'] == 0].shape)
proporcao = Buscatrain.loc[Buscatrain['is_fraud'] == 1].shape[0] / Buscatrain.shape[0]
print(proporcao)


(29750, 24)
(3261997, 24)
0.009037754116583079
(2720, 24)
(297280, 24)
0.009066666666666667


In [21]:
test = obtTest

X_train, y_train = obtTrain[features], obtTrain[target]
X_eva, Y_eval = obtEvaluation[features], obtEvaluation[target]
X_limiar, y_limiar = obtLimiar[features], obtLimiar[target]
X_test, y_test = test[features], test[target]


#conjunto menor para validar hiperparametros
X_busca = Buscatrain[features]
y_busca = Buscatrain[target]


In [22]:
import optuna
from sklearn.metrics import average_precision_score

In [24]:
%pip install torch torchvision torchaudio
import torch
print(torch.cuda.is_available())

Note: you may need to restart the kernel to use updated packages.
True


In [25]:
SEED = 42
N_INIT_TRIALS = 10
N_TRIALS = 30

X_busca = X_busca.copy()
X_eva = X_eva.copy()

for coluna in cat_features:
    X_busca[coluna] = X_busca[coluna].astype("category")
    X_eva[coluna] = X_eva[coluna].astype("category")

fixed_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
}

def objective(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.20, log=True
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1.0, 20.0, log=True
        ),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 100.0, log=True
        ),
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 1.0, 20.0, log=True
        ),
    }

    model = xgb.XGBClassifier(**fixed_params, **params)

    model.fit(
        X_busca,
        y_busca,
        eval_set=[(X_eva, Y_eval)],
        verbose=False,
    )

    probabilidades = model.predict_proba(X_eva)[:, 1]

    limiar_validacao, lucro = encontrar_melhor_limiar(
        model,
        X_eva,
        Y_eval,
    )

    prauc = average_precision_score(Y_eval, probabilidades)

    trial.set_user_attr("n_trees", model.n_estimators)
    trial.set_user_attr("limiar_validacao", limiar_validacao)
    trial.set_user_attr("prauc", prauc)

    return lucro


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=N_INIT_TRIALS,
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\nMelhor lucro de validação:", study.best_value)
print("Melhores hiperparâmetros:", study.best_params)
print("Número de árvores:", study.best_trial.user_attrs["n_trees"])
print(
    "Limiar usado na validação:",
    study.best_trial.user_attrs["limiar_validacao"],
)
print("PRAUC de diagnóstico:", study.best_trial.user_attrs["prauc"])

[I 2026-11-02 00:29:35,538] A new study created in memory with name: no-name-aa880e45-086b-45cb-9c01-2f1a770b8d6f


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-02 00:29:44,208] Trial 0 finished with value: 996719.25 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'min_child_weight': 8.960785365368121, 'gamma': 2.993292420985183, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 21.42302175774105, 'scale_pos_weight': 6.054365855469246}. Best is trial 0 with value: 996719.25.
[I 2026-11-02 00:29:53,937] Trial 1 finished with value: 1007626.875 and parameters: {'max_depth': 7, 'learning_rate': 0.010636066512540286, 'min_child_weight': 18.276027831785726, 'gamma': 4.162213204002109, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 0.033205591037519584, 'scale_pos_weight': 4.816414530907083}. Best is trial 1 with value: 1007626.875.
[I 2026-11-02 00:30:02,679] Trial 2 finished with value: 1007398.5 and parameters: {'max_depth': 6, 'learning_rate': 0.023927528765580644,

In [26]:
melhores_parametros = study.best_params.copy()

modelo_final = xgb.XGBClassifier(
    **fixed_params,
    **melhores_parametros,
    n_estimators=study.best_trial.user_attrs["n_trees"],
)

modelo_final.fit(
    X_train,
    y_train,
    verbose=False,
)


limiar_final, lucro_limiar = encontrar_melhor_limiar(
    modelo_final,
    X_limiar,
    y_limiar,
)

print("Melhor limiar:", limiar_final)
print("Lucro:", lucro_limiar)

Melhor limiar: 0.26673367619514465
Lucro: 253822.25


In [27]:
from sklearn.metrics import confusion_matrix,precision_score,recall_score

In [28]:

probabilidades_teste = modelo_final.predict_proba(X_test)[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_final).astype(int)

resultado_teste = X_test.copy()

resultado_teste["probabilidade_fraude"] = probabilidades_teste
resultado_teste["fraude_prevista"] = predicoes_teste
resultado_teste["fraude_real"] = y_test.to_numpy()

TransacoesAceitas, TransacoesRecusadas = classificarTransacoes(
    X_test,
    y_test,
    probabilidades_teste,
    limiar_final,
)

lucro_teste = calcular_lucro(TransacoesAceitas)
matriz = confusion_matrix(y_test, predicoes_teste)
precisao = precision_score(y_test, predicoes_teste, zero_division=0)
recall = recall_score(y_test, predicoes_teste, zero_division=0)
prauc_teste = average_precision_score(y_test, probabilidades_teste)

print("Matriz de confusão:")
print(matriz)
print("Lucro em maio:", lucro_teste)
print("Taxa de transações recusadas:", len(TransacoesRecusadas) / len(X_test))

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 0)
]
print("erro / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 1)
]
res
print("certo / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")


CALCULANDO MELHOR LUCRO...
VALOR:1637497.125
Matriz de confusão:
[[1030867   10951]
 [  10272    2962]]
Lucro em maio: 1637497.12
Taxa de transações recusadas: 0.01318702774839534
erro / total
10272/13234
certo / total
2962/13234


In [29]:
for parametro, valor in melhores_parametros.items():
    print(f"{parametro}: {valor}")

max_depth: 7
learning_rate: 0.013334893004636274
min_child_weight: 4.072618000736444
gamma: 4.951412343474423
subsample: 0.7747428778377763
colsample_bytree: 0.6068821606896359
reg_alpha: 9.573869651654385e-06
reg_lambda: 0.004413525334025521
scale_pos_weight: 9.447296535656582


In [30]:
# O limiar configurado também fica salvo dentro do modelo.
modelo_final.save_model("XGBoostingModel")


/usr/local/lib/python3.13/dist-packages/xgboost/sklearn.py:1183: UserWarning: [00:35:46] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1553: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
